In [3]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

146

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

gemini_client = genai.Client(
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            initial_delay=20.0, 
            attempts=3          
        )
    )
) # picks up the API key from the env variable GEMINI_API_KEY

In [10]:
import json
user_prompt = json.dumps(doc)

In [11]:
response = gemini_client.models.generate_content(
    model="gemini-3.6-flash",
    contents=user_prompt, # user_prompt
    config=types.GenerateContentConfig(
        system_instruction=data_gen_instructions, # for instructions, Gemini uses system_instruction in the config
        response_mime_type="application/json",
        response_schema=Questions
    )
)

In [12]:
result = response.parsed
print(result)

questions=['Is it too late for me to register and still get certified at the end?', "I'm late to the party, can I still enroll and turn in the final project?", 'Can I jump into the class now and still qualify for a completion certificate?', 'Am I able to sign up midway through and get credit if I finish the project on time?', 'If I start the course today, is there still a chance to earn the certificate?']


In [13]:
print(result.questions)

['Is it too late for me to register and still get certified at the end?', "I'm late to the party, can I still enroll and turn in the final project?", 'Can I jump into the class now and still qualify for a completion certificate?', 'Am I able to sign up midway through and get credit if I finish the project on time?', 'If I start the course today, is there still a chance to earn the certificate?']


In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    gemini_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Hey, is it too late to enroll in this cohort if I still want to get certified?', 'Can I sign up now and still be eligible for the final certificate?', 'I am late to the class, will I get the completion cert if I turn in the main assignment on time?', 'Is registration still open for people who want to earn the course credential?', 'If I start today, can I still qualify for the certificate by submitting the project before the cutoff?']


In [17]:
usage.prompt_token_count, usage.candidates_token_count

(174, 98)

In [ ]:
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.0001305, 'output_cost': 0.0003675, 'total_cost': 0.000498}

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q, # the question generated by the LLM
        "document": doc["id"] # the ID of the FAQ document that should answer the question
    })

records

[{'question': 'Hey, is it too late to enroll in this cohort if I still want to get certified?',
  'document': '74eb249bbf'},
 {'question': 'Can I sign up now and still be eligible for the final certificate?',
  'document': '74eb249bbf'},
 {'question': 'I am late to the class, will I get the completion cert if I turn in the main assignment on time?',
  'document': '74eb249bbf'},
 {'question': 'Is registration still open for people who want to earn the course credential?',
  'document': '74eb249bbf'},
 {'question': 'If I start today, can I still qualify for the certificate by submitting the project before the cutoff?',
  'document': '74eb249bbf'}]